# Introduction to Responsible AI

This notebook walks through exploring a real-life dataset and identifying sources of bias and unfairness in a trained model. We train a deep neural network classifier on the [UCI Adult Census Income](https://archive.ics.uci.edu/ml/datasets/adult) dataset to predict whether a person earns more than \$50K/year. We then use **[Fairlearn](https://fairlearn.org/)** — Microsoft's open-source fairness assessment and mitigation library — to analyze how the model performs across different demographic groups, and to apply standard fairness mitigation strategies.

> **Note:** This notebook originally used Google's *What-If Tool* (WIT). The What-If Tool's Python widget (`witwidget`) was archived by Google in 2023 and is no longer functional. We've rewritten the analysis using Fairlearn, which is actively maintained and provides the same conceptual fairness analyses programmatically.

**This notebook has 3 parts:**
* **Part 1** is dedicated to setting up the UCI Adult dataset, training the DNN classifier, and analyzing its fairness with Fairlearn across demographic groups.
* **Part 2** walks through interpreting the results — comparing the baseline single-threshold predictions against two standard fairness-mitigation strategies (demographic parity and equalized odds / equal opportunity).
* Finally, **Part 3** has a short knowledge check for self-evaluation.


# Part 1: Train a classifier and analyze its fairness

## Section A: Set up the UCI Adult dataset and train the DNN classifier

### Install required packages

> **First-run note for Colab users:**
> On the *first* run of the cell below in a fresh Colab session, the kernel installs `tensorflow`, `tf_keras`, `fairlearn`, `scipy`, and `scikit-learn`. After the install finishes, **you must restart the runtime** (Runtime → Restart session) and re-run the cell. This is because Colab's kernel has the old `scipy`/`scikit-learn` C extensions loaded in memory; only a restart picks up the freshly installed versions.


In [ ]:
import os
import subprocess
import sys

# Use Keras 2 (the API the course code was written against)
os.environ["TF_USE_LEGACY_KERAS"] = "1"

# Detect what's needed and install in one shot if anything is missing,
# pinning scipy/scikit-learn together to avoid the _lazywhere import error
# that occurs when fairlearn pulls in an sklearn incompatible with Colab's scipy.
try:
    import tensorflow as tf
    import tf_keras
    import fairlearn
    import sklearn
    import scipy
    print(f"TensorFlow {tf.__version__} (tf_keras {tf_keras.__version__})")
    print(f"fairlearn {fairlearn.__version__}")
    print(f"scikit-learn {sklearn.__version__}, scipy {scipy.__version__}")
except ImportError:
    print("Installing TensorFlow + Keras 2 + Fairlearn + matched scipy/sklearn...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
                           "tensorflow", "tf_keras", "fairlearn",
                           "scipy", "scikit-learn"])
    print()
    print("=" * 60)
    print("Install complete.")
    print("Now: Runtime > Restart session, then re-run this cell.")
    print("=" * 60)


### Load and pre-process the UCI Adult dataset

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf

csv_path = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
csv_columns = [
    "Age", "Workclass", "fnlwgt", "Education", "Education-Num", "Marital-Status",
    "Occupation", "Relationship", "Race", "Sex", "Capital-Gain", "Capital-Loss",
    "Hours-per-week", "Country", "Over-50K"
]

df = pd.read_csv(csv_path, names=csv_columns, skipinitialspace=True)
df["Over-50K"] = (df["Over-50K"] == ">50K").astype(int)

input_features = [
    "Age", "Workclass", "Education", "Marital-Status", "Occupation",
    "Relationship", "Race", "Sex", "Capital-Gain", "Capital-Loss",
    "Hours-per-week", "Country",
]

print(f"Loaded {len(df)} rows.")
df.head()


In [ ]:
# One-hot encode categoricals, normalize numerics, and split into train/test.
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

sensitive_sex = df["Sex"].values
sensitive_race = df["Race"].values

X = df[input_features].copy()
y = df["Over-50K"].values.astype(np.float32)

X = pd.get_dummies(X, drop_first=False)

numeric_cols = ["Age", "Capital-Gain", "Capital-Loss", "Hours-per-week"]
scaler = StandardScaler()
X[numeric_cols] = scaler.fit_transform(X[numeric_cols])
X = X.astype(np.float32).values

(X_train, X_test,
 y_train, y_test,
 sex_train, sex_test,
 race_train, race_test) = train_test_split(
    X, y, sensitive_sex, sensitive_race,
    test_size=0.2, random_state=42, stratify=y,
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")


### Build and train the DNN classifier

In [ ]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Input

num_epochs = 10  #@param {type: "number"}

model = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(128, activation="relu"),
    Dense(64, activation="relu"),
    Dense(32, activation="relu"),
    Dense(1, activation="sigmoid"),
])

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

model.fit(
    X_train, y_train,
    epochs=num_epochs, batch_size=128, validation_split=0.1, verbose=1,
)

loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f"\nOverall test accuracy: {acc:.3f}")


## Section B: Analyze fairness with Fairlearn

In [ ]:
y_pred_proba = model.predict(X_test, verbose=0).flatten()
y_pred = (y_pred_proba > 0.5).astype(int)


In [ ]:
from fairlearn.metrics import (
    MetricFrame, count, selection_rate,
    demographic_parity_difference, equalized_odds_difference,
)
from sklearn.metrics import accuracy_score, precision_score, recall_score

metric_frame = MetricFrame(
    metrics={
        "accuracy": accuracy_score,
        "precision": precision_score,
        "recall": recall_score,
        "selection_rate": selection_rate,
        "count": count,
    },
    y_true=y_test, y_pred=y_pred, sensitive_features=sex_test,
)

print("Metrics sliced by Sex:")
print(metric_frame.by_group.round(3))

print(f"\nDemographic parity difference: {demographic_parity_difference(y_test, y_pred, sensitive_features=sex_test):.3f}")
print(f"Equalized odds difference:      {equalized_odds_difference(y_test, y_pred, sensitive_features=sex_test):.3f}")


In [ ]:
import matplotlib.pyplot as plt

metric_frame.by_group.drop(columns=["count"]).plot(
    kind="bar", subplots=True, layout=(1, 4),
    figsize=(16, 4), legend=False, rot=0,
)
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import confusion_matrix

for group in np.unique(sex_test):
    mask = (sex_test == group)
    cm = confusion_matrix(y_test[mask], y_pred[mask])
    n = mask.sum()
    print(f"\n--- {group} (n = {n}) ---")
    print(f"             Predicted <=50K   Predicted >50K")
    print(f"True <=50K        {cm[0, 0]:6d}            {cm[0, 1]:6d}")
    print(f"True >50K         {cm[1, 0]:6d}            {cm[1, 1]:6d}")


## Section C: Mitigate bias with different fairness strategies

Fairlearn provides `ThresholdOptimizer`, which finds per-group prediction thresholds that satisfy different fairness criteria. We'll compare three strategies:

1. **Single threshold** — baseline: use the same 0.5 threshold for everyone (what we did above).
2. **Demographic parity** — enforce that the same fraction of each group is classified as positive.
3. **Equalized odds (≈ equal opportunity)** — enforce equal true-positive rates and equal false-positive rates across groups.


In [ ]:
import warnings
from fairlearn.postprocessing import ThresholdOptimizer
from sklearn.base import BaseEstimator, ClassifierMixin

# Silence Fairlearn's pandas dtype FutureWarning (cosmetic; internal to fairlearn).
warnings.filterwarnings("ignore", category=FutureWarning, module="fairlearn.*")


class KerasWrapper(BaseEstimator, ClassifierMixin):
    """Wrap a pre-trained Keras model so Fairlearn can treat it as an sklearn classifier."""

    def __init__(self, keras_model):
        self.keras_model = keras_model

    # Tell sklearn we are already fitted (we wrap a pre-trained model).
    def __sklearn_is_fitted__(self):
        return True

    @property
    def classes_(self):
        return np.array([0, 1])

    def fit(self, X, y, **kwargs):
        return self  # already trained

    def predict(self, X):
        return (self.keras_model.predict(X, verbose=0).flatten() > 0.5).astype(int)

    def predict_proba(self, X):
        proba = self.keras_model.predict(X, verbose=0).flatten()
        return np.column_stack([1 - proba, proba])


wrapped = KerasWrapper(model)


def evaluate_strategy(name, y_true, y_pred, sensitive):
    print(f"\n=== {name} ===")
    mf = MetricFrame(
        metrics={
            "accuracy": accuracy_score,
            "recall": recall_score,
            "selection_rate": selection_rate,
        },
        y_true=y_true, y_pred=y_pred, sensitive_features=sensitive,
    )
    print(mf.by_group.round(3))
    print(f"Demographic parity diff: {demographic_parity_difference(y_true, y_pred, sensitive_features=sensitive):.3f}")
    print(f"Equalized odds diff:     {equalized_odds_difference(y_true, y_pred, sensitive_features=sensitive):.3f}")


# 1. Baseline: single threshold (already computed)
evaluate_strategy("Single threshold (baseline)", y_test, y_pred, sex_test)

# 2. Demographic parity
opt_dp = ThresholdOptimizer(
    estimator=wrapped, constraints="demographic_parity",
    objective="balanced_accuracy_score", prefit=True,
)
opt_dp.fit(X_train, y_train, sensitive_features=sex_train)
y_pred_dp = opt_dp.predict(X_test, sensitive_features=sex_test)
evaluate_strategy("Demographic parity", y_test, y_pred_dp, sex_test)

# 3. Equalized odds (analog of the original notebook's Equal Opportunity)
opt_eo = ThresholdOptimizer(
    estimator=wrapped, constraints="equalized_odds",
    objective="balanced_accuracy_score", prefit=True,
)
opt_eo.fit(X_train, y_train, sensitive_features=sex_train)
y_pred_eo = opt_eo.predict(X_test, sensitive_features=sex_test)
evaluate_strategy("Equalized odds (Equal Opportunity)", y_test, y_pred_eo, sex_test)


# Part 2: Interpreting the results

Look at the three output blocks from the cell above and compare them:

1. **Single threshold (baseline)** — using a single 0.5 cutoff for both groups, you should see that the **selection rate** (fraction predicted as earning >\$50K) is substantially higher for `Male` than for `Female`. This reflects the underlying class imbalance in the dataset: ~30% of men earn >\$50K vs. ~11% of women.

2. **Demographic parity** — enforces equal selection rates. After this adjustment, the same percentage of men and women are predicted as high-income. But this can be *unfair* in a different sense: it forces the model to approve more women regardless of ground truth, raising false positives for women and lowering true positives for men.

3. **Equalized odds (Equal Opportunity)** — enforces equal true-positive rates across groups (and equal false-positive rates). The percentage of *deserving* (`>50K` ground truth) men and women who are correctly identified ends up the same — regardless of how many of each group there are in the population.

**Why this matters:** if this model were being used to approve loans based on predicted income, "single threshold" would systematically disadvantage women, "demographic parity" would harm both groups in different ways, and "equalized odds" gives deserving individuals an equal chance regardless of gender.

> The notion of *fairness* depends on context. For loan-approval-style decisions where the goal is to give qualified people equal opportunity, **equalized odds** is often the most defensible choice. For tasks where representation in the outcome matters (e.g. who gets shown a job ad), **demographic parity** may be appropriate. There is no single best fairness metric — choose the one that matches your application's goals.


# Part 3: Learning check

### Use the questions below to self-evaluate your learnings!

1. Which one of the following is the fairest threshold optimization strategy for this dataset?
   a. Single threshold
   b. Demographic parity
   c. Equal opportunity (equalized odds)
   d. Equal accuracy

2. Which of the following are ways of mitigating bias in a dataset/model?
   a. Increasing representation for specific social groups
   b. Increasing overall accuracy
   c. Adaptive thresholding
   d. Avoiding overfitting


### Expand this section to view the answers

1. **c**

2. **a, c, d**